In [ ]:
from pathlib import Path
import pandas as pd

ALLOWED_EXTENSIONS = {".csv", ".xlsx", ".xls"}


def get_file_path(file_path=None):
    if file_path is None:
        file_path = input("Enter file path: ")
    return str(file_path).strip().strip('"').strip("'")


def validate_file(file_path):
    path = Path(file_path)

    if not path.exists():
        print("File not found.")
        return False

    if not path.is_file():
        print("Path is not a file.")
        return False

    if path.suffix.lower() not in ALLOWED_EXTENSIONS:
        print(f"Unsupported file type: {path.suffix}. Allowed: {', '.join(sorted(ALLOWED_EXTENSIONS))}")
        return False

    return True


def load_data(file_path):
    path = Path(file_path)
    ext = path.suffix.lower()

    try:
        if ext == ".csv":
            try:
                return pd.read_csv(path)
            except UnicodeDecodeError:
                return pd.read_csv(path, encoding="latin-1")
        return pd.read_excel(path)

    except pd.errors.EmptyDataError:
        print("The file is empty.")
    except Exception as e:
        print(f"Could not read file: {e}")

    return None


def show_summary(df):
    line = "=" * 50

    print(f"\n{line}")
    print("  DATA OVERVIEW")
    print(line)
    print(f"  Rows           : {df.shape[0]}")
    print(f"  Columns        : {df.shape[1]}")
    print(f"  Duplicate rows : {df.duplicated().sum()}")

    print(f"\n{line}")
    print("  FIRST 5 ROWS")
    print(line)
    with pd.option_context("display.max_columns", None, "display.width", 120):
        print(df.head().T)

    print(f"\n{line}")
    print("  COLUMNS SUMMARY")
    print(line)
    summary = pd.DataFrame({
        "Type": df.dtypes.astype(str),
        "Missing": df.isnull().sum(),
        "Missing %": (df.isnull().mean() * 100).round(1),
        "Unique": df.nunique(),
    })
    print(summary.to_string())
    print()


def run_inspector(file_path=None):
    while True:
        cleaned_path = get_file_path(file_path)
        if validate_file(cleaned_path):
            break
        if file_path is not None:
            return None

    df = load_data(cleaned_path)
    if df is None:
        return None

    show_summary(df)
    return df


if __name__ == "__main__":
    run_inspector()